# Sentiment-Driven Market Analysis  
## Twitter Financial News Sentiment (Kaggle Notebook)

This notebook performs:
- dataset loading and inspection
- pragmatic preprocessing
- exploratory visualizations
- four-model comparison
- evaluation with paper-ready metrics
- error analysis and novelty-oriented hybrid inference

## 1. Objective

Predict financial tweet sentiment using the Hugging Face dataset:

- **Input column:** `text`
- **Target column:** `label`
- **Classes:** `0 = Bearish`, `1 = Bullish`, `2 = Neutral`

The notebook compares:
1. TF-IDF + Logistic Regression  
2. TF-IDF + Linear SVM  
3. Fine-tuned BERT-base  
4. Fine-tuned FinBERT + pragmatic rule-based gating (proposed hybrid)

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn emoji seaborn wordcloud

In [ ]:
import os
import re
import gc
import math
import json
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

import emoji
import torch
from torch.utils.data import Dataset as TorchDataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Load dataset from Hugging Face
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
dataset

In [ ]:
# Convert to pandas for EDA and classical ML
train_df = dataset["train"].to_pandas()
valid_df = dataset["validation"].to_pandas()

print(train_df.head())
print(train_df.columns.tolist())
print(train_df.shape, valid_df.shape)

In [ ]:
LABEL_MAP = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

train_df["label_name"] = train_df["label"].map(LABEL_MAP)
valid_df["label_name"] = valid_df["label"].map(LABEL_MAP)

display(train_df.sample(5, random_state=SEED))

In [ ]:
# Basic dataset checks
print("Train label counts:")
print(train_df["label_name"].value_counts())

print("Validation label counts:")
print(valid_df["label_name"].value_counts())

In [ ]:
# --- Exploratory visualizations ---

plt.figure(figsize=(7, 4))
train_df["label_name"].value_counts().reindex(["Bearish", "Bullish", "Neutral"]).plot(kind="bar")
plt.title("Training Label Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
valid_df["label_name"].value_counts().reindex(["Bearish", "Bullish", "Neutral"]).plot(kind="bar")
plt.title("Validation Label Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

train_df["text_len"] = train_df["text"].astype(str).str.len()
valid_df["text_len"] = valid_df["text"].astype(str).str.len()

plt.figure(figsize=(7, 4))
plt.hist(train_df["text_len"], bins=40)
plt.title("Training Tweet Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(valid_df["text_len"], bins=40)
plt.title("Validation Tweet Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Optional preview of representative examples
for lbl in ["Bearish", "Bullish", "Neutral"]:
    print(f"\n=== {lbl} examples ===")
    examples = train_df[train_df["label_name"] == lbl]["text"].head(3).tolist()
    for ex in examples:
        print("-", ex)

## 2. Pragmatic preprocessing

Two text variants are created:

- **`text_basic`** for classical ML models
- **`text_pragmatic`** for transformer and hybrid analysis

The pragmatic version preserves sentiment-bearing signals such as:
- punctuation emphasis
- repeated letters
- uppercase emphasis
- emojis
- contrast markers (`but`, `however`, `although`)
- negation cues
- hedge markers (`if`, `may`, `could`)

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
MULTI_SPACE_RE = re.compile(r"\s+")
REPEAT_PUNCT_RE = re.compile(r"([!?.,]){2,}")
REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}")
NUMBER_RE = re.compile(r"\b\d+([.,]\d+)?\b")

EMOJI_MAP = {
    "📈": "emoji_bullish",
    "🚀": "emoji_bullish",
    "🔥": "emoji_bullish",
    "💹": "emoji_bullish",
    "📉": "emoji_bearish",
    "💥": "emoji_bearish",
    "😡": "emoji_bearish",
    "😐": "emoji_neutral",
    "🤔": "emoji_hedge",
    "😂": "emoji_sarcasm",
    "🙃": "emoji_sarcasm",
}

CONTRAST_WORDS = {"but", "however", "although", "though", "yet", "while", "despite"}
HEDGE_WORDS = {"if", "may", "might", "could", "would", "possible", "possibly", "seems", "appears", "could"}
NEGATION_WORDS = {"no", "not", "never", "none", "without", "hardly", "neither", "nor"}

POS_WORDS = {
    "beat","beats","beating","surge","surges","up","gain","gains","bullish","upgrade","upgraded",
    "strong","record","profit","growth","buy","outperform","raise","raises","raised","higher","rally",
    "optimistic","good","great","positive","bounce","bull","spike","expands","expansion","improve","improves"
}
NEG_WORDS = {
    "miss","misses","missed","down","loss","losses","bearish","downgrade","downgraded","weak","decline",
    "fall","falls","cut","cuts","lower","short","risk","lawsuit","drop","plunge","negative","bad","concern",
    "warning","warns","warned","slump","bear"
}

def replace_emojis(text: str) -> str:
    for e, token in EMOJI_MAP.items():
        text = text.replace(e, f" {token} ")
    return text

def clean_basic(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(r"\1", text)
    text = replace_emojis(text)
    text = text.lower()
    text = NUMBER_RE.sub(" <NUM> ", text)
    text = REPEAT_PUNCT_RE.sub(r" \1 ", text)
    text = REPEAT_CHAR_RE.sub(r"\1\1", text)
    text = re.sub(r"[^a-z<>()!?.,\s_]", " ", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text

def clean_pragmatic(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" <URL> ", text)
    text = MENTION_RE.sub(" <USER> ", text)
    text = HASHTAG_RE.sub(r" \1 ", text)
    text = replace_emojis(text)
    text = NUMBER_RE.sub(" <NUM> ", text)
    text = REPEAT_PUNCT_RE.sub(r" \1 ", text)
    text = REPEAT_CHAR_RE.sub(r"\1\1", text)
    text = MULTI_SPACE_RE.sub(" ", text).strip()
    return text

def tokenize_simple(text: str):
    return re.findall(r"[A-Za-z_<>]+|[!?.,]", text.lower())

def rule_based_sentiment(text: str):
    tokens = tokenize_simple(text)
    token_set = set(tokens)
    pos_hits = len(token_set & POS_WORDS)
    neg_hits = len(token_set & NEG_WORDS)
    has_contrast = any(w in token_set for w in CONTRAST_WORDS)
    has_hedge = any(w in token_set for w in HEDGE_WORDS)
    has_negation = any(w in token_set for w in NEGATION_WORDS)
    has_sarcasm = ("emoji_sarcasm" in text.lower()) or ('"' in text) or ("..." in text) or ("!? " in text)

    # Very light clause-based heuristic for "X but Y"
    lower = text.lower()
    if any(c in lower for c in CONTRAST_WORDS):
        for c in ["but", "however", "although", "though", "yet"]:
            if c in lower:
                parts = lower.split(c, 1)
                left = parts[0]
                right = parts[1]
                left_pos = sum(w in left for w in POS_WORDS)
                left_neg = sum(w in left for w in NEG_WORDS)
                right_pos = sum(w in right for w in POS_WORDS)
                right_neg = sum(w in right for w in NEG_WORDS)
                if right_neg > right_pos:
                    return "Bearish", 0.90, {"contrast": True, "hedge": has_hedge, "sarcasm": has_sarcasm}
                if right_pos > right_neg:
                    return "Bullish", 0.90, {"contrast": True, "hedge": has_hedge, "sarcasm": has_sarcasm}

    score = pos_hits - neg_hits
    if has_negation:
        score = -score if score != 0 else score

    if has_hedge and abs(score) <= 1:
        return "Neutral", 0.85, {"contrast": has_contrast, "hedge": True, "sarcasm": has_sarcasm}

    if score > 0:
        return "Bullish", min(0.55 + 0.1 * score, 0.95), {"contrast": has_contrast, "hedge": has_hedge, "sarcasm": has_sarcasm}
    if score < 0:
        return "Bearish", min(0.55 + 0.1 * abs(score), 0.95), {"contrast": has_contrast, "hedge": has_hedge, "sarcasm": has_sarcasm}
    return "Neutral", 0.60, {"contrast": has_contrast, "hedge": has_hedge, "sarcasm": has_sarcasm}

train_df["text_basic"] = train_df["text"].apply(clean_basic)
train_df["text_pragmatic"] = train_df["text"].apply(clean_pragmatic)
valid_df["text_basic"] = valid_df["text"].apply(clean_basic)
valid_df["text_pragmatic"] = valid_df["text"].apply(clean_pragmatic)

print(train_df[["text", "text_basic", "text_pragmatic"]].head(3).to_string(index=False))

In [ ]:
# Preprocessing visual sanity check
sample_rows = train_df.sample(5, random_state=SEED)[["text", "text_basic", "text_pragmatic", "label_name"]]
for _, row in sample_rows.iterrows():
    print("\nLABEL:", row["label_name"])
    print("RAW :", row["text"])
    print("BASIC:", row["text_basic"])
    print("PRAG :", row["text_pragmatic"])

## 3. Classical baselines

These models provide strong and interpretable benchmarks:
- TF-IDF + Logistic Regression
- TF-IDF + Linear SVM

They are useful for showing the gain from transformer-based methods.

In [ ]:
X_train = train_df["text_basic"].fillna("").tolist()
y_train = train_df["label"].tolist()
X_valid = valid_df["text_basic"].fillna("").tolist()
y_valid = valid_df["label"].tolist()

def evaluate_predictions(y_true, y_pred, name="model"):
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    report = classification_report(y_true, y_pred, target_names=[LABEL_MAP[i] for i in sorted(LABEL_MAP)], output_dict=True, zero_division=0)
    return {
        "model": name,
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f1_weighted,
        "report": report,
    }

def fit_and_eval_classical(model, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    return pred, evaluate_predictions(y_valid, pred, name=name)

tfidf_logreg = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        min_df=2,
        sublinear_tf=True
    )),
    ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", n_jobs=None))
])

tfidf_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        min_df=2,
        sublinear_tf=True
    )),
    ("clf", LinearSVC(class_weight="balanced"))
])

logreg_pred, logreg_metrics = fit_and_eval_classical(tfidf_logreg, "TFIDF + Logistic Regression")
svm_pred, svm_metrics = fit_and_eval_classical(tfidf_svm, "TFIDF + Linear SVM")

print(logreg_metrics["model"], logreg_metrics["f1_macro"])
print(svm_metrics["model"], svm_metrics["f1_macro"])

## 4. Transformer fine-tuning

Two transformer models are trained:

- **BERT-base-uncased**: general language baseline
- **FinBERT**: finance-domain model

The code below is written to be Kaggle-friendly:
- small max sequence length
- early stopping
- mixed precision when available
- compact reporting after training

In [ ]:
class TweetDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    DataCollatorWithPadding,
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def train_transformer_ultra_fast(
    train_texts, train_labels, valid_texts, valid_labels,
    output_dir,
    model_name="distilbert-base-uncased",   # FIX 1: accept model_name as param
    epochs=3,
    lr=5e-4,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    # FIX 2: Generic encoder freeze – works for DistilBERT, BERT, FinBERT, etc.
    # AutoModel wraps the base encoder as the first child module.
    base_encoder = next(iter(model.children()))
    for param in base_encoder.parameters():
        param.requires_grad = False

    MAX_LEN = 64
    train_ds = TweetDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
    valid_ds = TweetDataset(valid_texts, valid_labels, tokenizer, max_len=MAX_LEN)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=64 if torch.cuda.is_available() else 8,
        per_device_eval_batch_size=64 if torch.cuda.is_available() else 16,
        num_train_epochs=epochs,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=1,
        logging_steps=100,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred.predictions, eval_pred.label_ids
        preds = np.argmax(logits, axis=1)
        _, _, f1_macro, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        return {"f1_macro": f1_macro}

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    trainer.train()

    # FIX 3 & 4: Run prediction → compute full metrics → return "metrics" + "pred_labels"
    out = trainer.predict(valid_ds)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=1).numpy()
    pred_labels = np.argmax(probs, axis=1).tolist()
    y_true = list(valid_labels)

    acc = accuracy_score(y_true, pred_labels)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )
    p_wt,  r_wt,  f1_wt,  _ = precision_recall_fscore_support(
        y_true, pred_labels, average="weighted", zero_division=0
    )

    metrics = {
        "model": model_name,
        "accuracy": acc,
        "precision_macro": p_mac,
        "recall_macro": r_mac,
        "f1_macro": f1_mac,
        "precision_weighted": p_wt,
        "recall_weighted": r_wt,
        "f1_weighted": f1_wt,
    }

    print(f"[{model_name}] Acc={acc:.4f}  F1-macro={f1_mac:.4f}  F1-weighted={f1_wt:.4f}")

    return {
        "trainer": trainer,
        "model": model,
        "tokenizer": tokenizer,
        "probs": probs,
        "pred_labels": pred_labels,   # FIX 4a: used by confusion matrix (cell 26)
        "metrics": metrics,           # FIX 4b: used by results table (cells 18, 22)
    }

# ── Train DistilBERT baseline ─────────────────────────────────────────────────
bert_results = train_transformer_ultra_fast(
    train_texts=train_df["text_pragmatic"].tolist(),
    train_labels=train_df["label"].tolist(),
    valid_texts=valid_df["text_pragmatic"].tolist(),
    valid_labels=valid_df["label"].tolist(),
    output_dir="./fast_bert_sentiment",
    model_name="distilbert-base-uncased",
)


In [ ]:
# ── Train FinBERT ─────────────────────────────────────────────────────────────
finbert_results = train_transformer_ultra_fast(
    model_name="ProsusAI/finbert",          # FIX: param now accepted
    train_texts=train_df["text_pragmatic"].tolist(),
    train_labels=train_df["label"].tolist(),
    valid_texts=valid_df["text_pragmatic"].tolist(),
    valid_labels=valid_df["label"].tolist(),
    output_dir="./finbert_sentiment",
    epochs=3,
    lr=2e-5,
)

# FIX: "metrics" key now exists in return dict
print("FinBERT macro-F1:", finbert_results["metrics"]["f1_macro"])


## 5. Proposed hybrid model: FinBERT + pragmatic gating

The proposed novelty layer combines:
- transformer confidence
- contrast/sarcasm/hedging cues
- rule-based override for ambiguous tweets

This is useful for examples where the literal wording and the implied market direction differ.

In [ ]:
# Helper for getting transformer probabilities from trained model
def predict_proba_transformer(trainer, dataset):
    out = trainer.predict(dataset)
    logits = out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    conf = probs.max(axis=1)
    return preds, probs, conf

finbert_tokenizer = finbert_results["tokenizer"]
finbert_trainer = finbert_results["trainer"]
finbert_valid_ds = TweetDataset(
    valid_df["text_pragmatic"].tolist(),
    valid_df["label"].tolist(),
    finbert_tokenizer,
    max_len=128
)

finbert_pred, finbert_probs, finbert_conf = predict_proba_transformer(finbert_trainer, finbert_valid_ds)

ID_TO_LABEL = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
LABEL_TO_ID = {v: k for k, v in ID_TO_LABEL.items()}

def hybrid_override(text, finbert_pred_id, confidence, threshold=0.65):
    rule_label, rule_conf, flags = rule_based_sentiment(text)
    rule_id = LABEL_TO_ID[rule_label]

    # Override when confidence is low or pragmatic cues are strong
    strong_cue = flags["contrast"] or flags["sarcasm"] or flags["hedge"]
    if confidence < threshold or strong_cue:
        return rule_id, {"rule_label": rule_label, "rule_conf": rule_conf, "flags": flags}
    return finbert_pred_id, {"rule_label": rule_label, "rule_conf": rule_conf, "flags": flags}

hybrid_pred = []
hybrid_meta = []

for txt, pred_id, conf in zip(valid_df["text_pragmatic"].tolist(), finbert_pred.tolist(), finbert_conf.tolist()):
    out_id, meta = hybrid_override(txt, pred_id, conf, threshold=0.65)
    hybrid_pred.append(out_id)
    hybrid_meta.append(meta)

hybrid_metrics = evaluate_predictions(y_valid, hybrid_pred, name="FinBERT + Pragmatic Gating")
hybrid_metrics["f1_macro"]

## 6. Results Table and Evaluation

### Final Model Comparison

The table below summarises all five models evaluated on the validation split. 
The primary ranking metric is **Macro F1**, which weights each class equally and 
therefore gives an unbiased view of performance across Bearish, Bullish, and Neutral.

| Model | Accuracy | Precision (Macro) | Recall (Macro) | F1 (Macro) | F1 (Weighted) |
|---|---|---|---|---|---|
| **TF-IDF + Linear SVM** ✅ | **0.8409** | **0.7895** | 0.7753 | **0.7821** | **0.8396** |
| TF-IDF + Logistic Regression | 0.8166 | 0.7470 | **0.7813** | 0.7620 | 0.8202 |
| FinBERT (frozen) | 0.7567 | 0.6829 | 0.7163 | 0.6960 | 0.7603 |
| FinBERT + Pragmatic Gating | 0.7688 | 0.7108 | 0.6729 | 0.6893 | 0.7634 |
| DistilBERT-base-uncased (frozen) | 0.7680 | 0.7114 | 0.6641 | 0.6649 | 0.7607 |

### Key Findings

**Winner — TF-IDF + Linear SVM** leads on four out of five metrics:

- **Highest accuracy (84.09%)** — correctly classifies the largest share of validation tweets.
- **Highest macro precision (0.7895)** — when it predicts a class, it is most likely to be right, 
even for minority classes.
- **Highest macro F1 (0.7821)** — the best balance between precision and recall across all three 
sentiment classes.
- **Highest weighted F1 (0.8396)** — strong performance even when class frequency is taken into account.

**Runner-up — TF-IDF + Logistic Regression** achieves the best macro recall (0.7813), meaning it 
misses the fewest true positives per class. However, it pays for this in lower precision (0.7470) 
and a Macro F1 gap of ~2 points versus the SVM.

**Transformer models underperform in the frozen setting.** All three transformer variants 
(FinBERT, DistilBERT, and the hybrid) score between 0.66–0.70 on Macro F1. This is expected: 
freezing the encoder body and training only the classification head severely limits the models' 
ability to adapt their representations to the financial-tweet domain. Full fine-tuning would 
likely close or reverse this gap.

**The hybrid FinBERT + Pragmatic Gating model** does not outperform frozen FinBERT on Macro F1 
(0.6893 vs 0.6960), suggesting that the rule-based override layer slightly hurts overall recall 
(0.6729 vs 0.7163) despite improving precision (0.7108 vs 0.6829). The pragmatic cues are most 
valuable in the error-analysis stage rather than as a blanket prediction gate.

### Why the Classical Model Wins Here

Several factors explain why TF-IDF + Linear SVM outperforms the frozen transformers on this dataset:

1. **Dataset size is moderate** — the Twitter Financial News Sentiment dataset is small enough 
that sparse bag-of-ngrams representations capture the discriminative vocabulary (e.g., *beat*, 
*miss*, *downgrade*) without overfitting.
2. **Financial jargon is lexically rich** — key sentiment is often carried by a handful of 
domain-specific tokens that a TF-IDF model with bigrams handles well.
3. **Frozen transformer heads cannot adapt** — without gradient flow into the encoder, the 
pretrained representations are only partially aligned with the financial-tweet distribution.

The key comparison metric for model selection and paper reporting is **Macro F1**, because:
- It treats all three classes (Bearish, Bullish, Neutral) with equal weight.
- It penalises models that achieve high accuracy by favouring the majority class.
- It is the standard metric in multi-class NLP benchmarks.

The notebook also reports accuracy, macro precision/recall, and weighted F1 for completeness, 
and confusion matrices are provided in Section 7 to expose class-level error patterns.

In [ ]:
# ── FIX: use friendly display names and access now-valid "metrics" keys ──────
bert_m    = bert_results["metrics"]
finbert_m = finbert_results["metrics"]

results = pd.DataFrame([
    {
        "Model":              logreg_metrics["model"],
        "Accuracy":           logreg_metrics["accuracy"],
        "Precision (Macro)":  logreg_metrics["precision_macro"],
        "Recall (Macro)":     logreg_metrics["recall_macro"],
        "F1 (Macro)":         logreg_metrics["f1_macro"],
        "F1 (Weighted)":      logreg_metrics["f1_weighted"],
    },
    {
        "Model":              svm_metrics["model"],
        "Accuracy":           svm_metrics["accuracy"],
        "Precision (Macro)":  svm_metrics["precision_macro"],
        "Recall (Macro)":     svm_metrics["recall_macro"],
        "F1 (Macro)":         svm_metrics["f1_macro"],
        "F1 (Weighted)":      svm_metrics["f1_weighted"],
    },
    {
        "Model":              "DistilBERT-base-uncased (frozen)",
        "Accuracy":           bert_m["accuracy"],
        "Precision (Macro)":  bert_m["precision_macro"],
        "Recall (Macro)":     bert_m["recall_macro"],
        "F1 (Macro)":         bert_m["f1_macro"],
        "F1 (Weighted)":      bert_m["f1_weighted"],
    },
    {
        "Model":              "FinBERT (frozen)",
        "Accuracy":           finbert_m["accuracy"],
        "Precision (Macro)":  finbert_m["precision_macro"],
        "Recall (Macro)":     finbert_m["recall_macro"],
        "F1 (Macro)":         finbert_m["f1_macro"],
        "F1 (Weighted)":      finbert_m["f1_weighted"],
    },
    {
        "Model":              hybrid_metrics["model"],
        "Accuracy":           hybrid_metrics["accuracy"],
        "Precision (Macro)":  hybrid_metrics["precision_macro"],
        "Recall (Macro)":     hybrid_metrics["recall_macro"],
        "F1 (Macro)":         hybrid_metrics["f1_macro"],
        "F1 (Weighted)":      hybrid_metrics["f1_weighted"],
    },
]).sort_values("F1 (Macro)", ascending=False).reset_index(drop=True)

display(results.style.format({c: "{:.4f}" for c in results.columns if c != "Model"})
               .highlight_max(subset=[c for c in results.columns if c != "Model"],
                              color="lightgreen"))


In [ ]:
# Save results for paper tables
results.to_csv("model_comparison_results.csv", index=False)
results

## 7. Visualizing model comparison

These plots are suitable for a paper or presentation:
- macro F1 comparison
- accuracy comparison
- confusion matrices

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(results["Model"], results["F1 (Macro)"])
plt.title("Model Comparison by Macro F1")
plt.xlabel("Macro F1")
plt.ylabel("Model")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.barh(results["Model"], results["Accuracy"])
plt.title("Model Comparison by Accuracy")
plt.xlabel("Accuracy")
plt.ylabel("Model")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[LABEL_MAP[i] for i in [0, 1, 2]],
                yticklabels=[LABEL_MAP[i] for i in [0, 1, 2]])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

plot_confusion(y_valid, logreg_pred,                       "Confusion Matrix: TF-IDF + Logistic Regression")
plot_confusion(y_valid, svm_pred,                          "Confusion Matrix: TF-IDF + Linear SVM")
plot_confusion(y_valid, bert_results["pred_labels"],       "Confusion Matrix: DistilBERT-base-uncased (frozen)")   # FIX: key now exists
plot_confusion(y_valid, finbert_results["pred_labels"],    "Confusion Matrix: FinBERT (frozen)")                   # FIX: added FinBERT CM
plot_confusion(y_valid, hybrid_pred,                       "Confusion Matrix: FinBERT + Pragmatic Gating")


## 8. Error analysis

Inspect the tweets the proposed model misclassified.  
This section supports the discussion of linguistic edge cases such as:
- sarcasm
- hedging
- contrastive clauses
- mixed sentiment
- market jargon

In [ ]:
hybrid_pred_names = [ID_TO_LABEL[i] for i in hybrid_pred]
true_names = [ID_TO_LABEL[i] for i in y_valid]

error_df = valid_df.copy()
error_df["true_label"] = true_names
error_df["pred_label"] = hybrid_pred_names
error_df["correct"] = error_df["true_label"] == error_df["pred_label"]
error_df["finbert_conf"] = finbert_conf
error_df["rule_label"] = [m["rule_label"] for m in hybrid_meta]
error_df["rule_conf"] = [m["rule_conf"] for m in hybrid_meta]

misclassified = error_df[~error_df["correct"]][["text", "true_label", "pred_label", "finbert_conf", "rule_label", "rule_conf"]]
display(misclassified.head(20))

In [ ]:
# Save misclassified samples for discussion in the paper
misclassified.to_csv("hybrid_misclassified_samples.csv", index=False)
print("Saved:", os.path.abspath("model_comparison_results.csv"))
print("Saved:", os.path.abspath("hybrid_misclassified_samples.csv"))

## 9. Paper-Ready Narrative Points

Use these in your report or presentation.

### Result Summary

**TF-IDF + Linear SVM is the best-performing model** across this experiment. 
It achieves 84.09% accuracy and a Macro F1 of 0.7821, outperforming all transformer variants 
evaluated in the frozen setting. The classical pipeline is also the most deployment-friendly: 
it is lightweight, fast at inference time, and requires no GPU.

### Novelty

- The proposed hybrid model adds a pragmatic reasoning layer on top of FinBERT to detect 
contrast markers, hedging language, and sarcasm cues — linguistic phenomena that pure classifiers 
typically miss.
- Confidence-based override (`threshold = 0.65`) allows the rule system to intervene when the 
transformer is uncertain, rather than blindly overriding all predictions.
- Despite not reaching the Macro F1 of the SVM baseline in the frozen setting, the pragmatic 
gating improves macro precision (+2.8 pp over frozen FinBERT), which is important for 
applications where false positives carry higher cost than false negatives.

### Why Macro F1 Matters

- Financial sentiment datasets often have unequal class distributions. 
Accuracy alone can be gamed by predicting the majority class.
- Macro F1 penalises models that ignore minority classes — critical for Bearish detection, 
where missing a negative signal can be costly in a market context.
- Weighted F1 is reported alongside Macro F1 to show the trade-off between class-balanced 
and frequency-weighted performance.

### Expected Contribution

- Demonstrates that a simple TF-IDF + SVM pipeline remains a strong, lightweight baseline 
for financial tweet sentiment, outperforming frozen transformers.
- Introduces a pragmatic gating mechanism that improves precision on ambiguous financial language.
- Provides a reproducible preprocessing and evaluation pipeline with two text variants 
(`text_basic` for classical ML, `text_pragmatic` for transformer / hybrid models).
- Includes full confusion matrices, class-wise metrics, and qualitative error analysis to 
support discussion of failure modes (sarcasm, hedging, contrastive clauses, market jargon).

### Limitations and Future Work

- The transformer models were evaluated in a **frozen** setting due to computational constraints. 
Full fine-tuning of FinBERT is expected to close the ~8 pp Macro F1 gap versus the SVM.
- The pragmatic rule system uses a manually curated lexicon. Extending it with domain-adaptive 
word embeddings or a learned confidence calibrator would likely improve the hybrid's recall.
- The dataset is English-only; cross-lingual generalisation is untested.

## 10. Next extension ideas

If you want to make the paper even stronger, you can add:
- ablation study: FinBERT vs FinBERT + rules
- statistical significance test on predictions
- SHAP / feature importance for classical baselines
- attention-based qualitative examples for transformers

## 11. Saving the Best Model for Deployment

Since **TF-IDF + Linear SVM** is the best model, we serialise the complete scikit-learn 
Pipeline (vectoriser + classifier) to disk using `joblib`. 
This single file is all that is needed for Streamlit inference — no GPU, no transformers library.

The saved artefacts are:
- `best_model_tfidf_svm.joblib` — the full pipeline (TfidfVectorizer + LinearSVC)
- `label_map.json` — mapping from integer label → sentiment string


In [ ]:
import joblib
import json
import os

# ── 1. Save the best pipeline ────────────────────────────────────────────────
MODEL_PATH = "best_model_tfidf_svm.joblib"
joblib.dump(tfidf_svm, MODEL_PATH)
print(f"Model saved → {os.path.abspath(MODEL_PATH)}")

# ── 2. Save the label map for Streamlit use ───────────────────────────────────
LABEL_MAP_PATH = "label_map.json"
with open(LABEL_MAP_PATH, "w") as f:
    json.dump(LABEL_MAP, f)   # {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
print(f"Label map saved → {os.path.abspath(LABEL_MAP_PATH)}")

# ── 3. Quick smoke-test ───────────────────────────────────────────────────────
loaded_model = joblib.load(MODEL_PATH)
test_tweets = [
    "$AAPL beats Q3 earnings estimates, revenue up 12%",
    "Markets plunge as Fed signals more rate hikes ahead",
    "Tesla maintained its market position with steady deliveries this quarter",
]
test_clean  = [clean_basic(t) for t in test_tweets]
predictions = loaded_model.predict(test_clean)

print("\nSmoke-test predictions:")
for tweet, pred in zip(test_tweets, predictions):
    print(f"  [{LABEL_MAP[pred]:>8}]  {tweet}")


### Streamlit Usage

In your Streamlit app, load and call the model with:

```python
import joblib
import json
import re

# ── helpers (copy clean_basic from this notebook) ──────────────────
# paste clean_basic() and its regex constants here

MODEL  = joblib.load("best_model_tfidf_svm.joblib")
LABELS = json.load(open("label_map.json"))   # keys are strings after JSON round-trip

def predict_sentiment(raw_text: str) -> str:
    cleaned = clean_basic(raw_text)
    pred    = MODEL.predict([cleaned])[0]
    return LABELS[str(pred)]   # 'Bearish' | 'Bullish' | 'Neutral'
```

> **Note:** `joblib.load` is safe here because we saved a scikit-learn pipeline with no 
external callbacks. The model file is typically 2–5 MB and loads in under a second.